# Data Cleaning

The purpose of this notebook is to match metadata to the MIDI files as downloaded and perform some cleaning.

In its raw state, the matched files from the Lakh Dataset do not have self-explanatory filenames. To match them to songs, they need to be paired up with the corresponding Million Song Dataset Metadata. The filenames do howveer match up here.

In [1]:
from pathlib import Path
import sys
import h5py
import pandas as pd
from tqdm import tqdm
    
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.config.paths import MSD_METADATA_DIR, PROCESSED_DATA_DIR

In [2]:
h5_files = list(MSD_METADATA_DIR.rglob("*.h5"))

output = set()

for f in tqdm(h5_files):
    with h5py.File(f, "r") as h5:
        if tuple(h5.keys()) not in output:
            print(h5.keys())  # check available datasets
            output.add(tuple(h5.keys()))
            for item in h5.keys():
                if item not in output:
                    print(h5[item].keys())  # check available datasets
                    output.add(item)

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
from tqdm import tqdm
import h5py


def print_h5_structure(obj, indent=0):
    prefix = "  " * indent

    for key, value in obj.items():
        print(f"{prefix}{key}")

        if isinstance(value, h5py.Group):
            print_h5_structure(value, indent + 1)

        elif isinstance(value, h5py.Dataset):
            print(f"{prefix}  shape={value.shape}, dtype={value.dtype}")


for f in tqdm(MSD_METADATA_DIR.rglob("*.h5")):
    print(f"\n{'=' * 80}")
    print(f)
    print('=' * 80)

    with h5py.File(f, "r") as h5:
        print_h5_structure(h5)

0it [00:00, ?it/s]


In [ ]:
from pathlib import Path
from tqdm import tqdm
import h5py

seen = set()

for f in tqdm(MSD_METADATA_DIR.rglob("*.h5")):
    with h5py.File(f, "r") as h5:

        def collect(name, obj):
            key = (name, type(obj).__name__)

            if key not in seen:
                seen.add(key)

                if isinstance(obj, h5py.Dataset):
                    print(
                        f"NEW DATASET: {name} "
                        f"(shape={obj.shape}, dtype={obj.dtype})"
                    )
                else:
                    print(f"NEW GROUP:   {name}")

        h5.visititems(collect)

0it [00:00, ?it/s]


In [6]:
def extract_h5_metadata(h5_path):
    with h5py.File(h5_path, "r") as h5:

        song = h5["metadata"]["songs"][0]

        return {
            "track_id": Path(h5_path).stem,
            "artist_name": song["artist_name"].decode("utf-8"),
            "artist_id": song["artist_id"].decode("utf-8"),
            "title": song["title"].decode("utf-8"),
            "song_id": song["song_id"].decode("utf-8"),
            "release": song["release"].decode("utf-8"),
            "path": h5_path
        }

In [7]:

h5_files = list(MSD_METADATA_DIR.rglob("*.h5"))

rows = []

for f in tqdm(h5_files):
    try:
        rows.append(extract_h5_metadata(f))
    except Exception as e:
        print(f"Failed: {f} -> {e}")

100%|██████████| 31034/31034 [09:00<00:00, 57.47it/s] 


In [8]:
df = pd.DataFrame(rows)
print(df.shape)
df.head()

(31034, 7)


,track_id,artist_name,artist_id,title,song_id,release,path
0,TRAAAGR128F425B14B,Cyndi Lauper,ARGE7G11187FB37E05,Into The Nightlife,SONRWUU12AF72A4283,Bring Ya To The Brink,C:\Users\danie\Documents\GitHub\music-represen...
1,TRAAAZF12903CCCF6B,Matthew Wilder,ARJJ8611187FB5321F,Break My Stride,SOUCVHW12AB018E830,I Don't Speak The Language,C:\Users\danie\Documents\GitHub\music-represen...
2,TRAABVM128F92CA9DC,Tesla,ARYKCQI1187FB3B18F,Caught In A Dream,SOXLBJT12A8C140925,Gold,C:\Users\danie\Documents\GitHub\music-represen...
3,TRAABXH128F42955D6,Brian Wilson,ARD9UVF1187B9B17FE,Keep An Eye On Summer (Album Version),SOHXFBA12A8C13D637,Imagination,C:\Users\danie\Documents\GitHub\music-represen...
4,TRAACQE12903CC706C,Old Man River,ARDDIBO1187B9B0822,Summer,SOGUCAN12AB017BF99,Good Morning,C:\Users\danie\Documents\GitHub\music-represen...


In [5]:
df.to_csv(f"{PROCESSED_DATA_DIR}/midi_index.csv", index=False)